# 07 — UMTTA Phase A: logit ImageNet untuk backbone kedua

**Kenapa notebook ini ada.** Klaim PCC sejauh ini berdiri di skor rilis CCC
(SimCLRv2 + linear probe). Untuk paper kita butuh dua hal yang hanya bisa didapat
dengan menghitung skor sendiri:

1. **Backbone kedua** — ResNet-50 tersupervisi, supaya klaim tidak bergantung satu
   model. Aturan keras `AGENTS.md`: baseline harus dihitung ulang pada backbone yang
   sama, tidak pernah dibandingkan lintas backbone.
2. **Pintu ke ImageNet-C** — pipeline yang sama, dengan direktori terkorupsi.

Repo UMTTA memisahkan **Phase A** (ekstrak + cache logit, GPU, lambat) dari
**Phase B** (evaluasi, numpy murni, cepat). Itu berarti α dan fungsi skor **gratis**:
yang mahal hanya dataset × seed × metode. Notebook ini menjalankan Phase A sekali,
menyimpan cache-nya ke Drive, lalu Phase B bisa diulang berapa kali pun tanpa GPU —
termasuk oleh PCC.

**Butuh GPU.** ~6,7 GB unduhan, ~13 GB disk sementara.

## 1. Config

In [ ]:
# === EDIT ME ===========================================================
DRIVE_ROOT = '/content/drive/MyDrive/pcc'
UMTTA_URL  = 'https://github.com/octadion/umtta_conformal.git'
UMTTA_DIR  = '/content/umtta'

VAL_TAR = 'https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar'
LBL_URL = ('https://github.com/tensorflow/models/raw/master/research/slim/'
           'datasets/imagenet_2012_validation_synset_labels.txt')
VAL_DIR = '/content/imagenet_val'      # ImageFolder root, satu subfolder per WNID

# Protokol sama dengan paper lama (supplementary G): 25.000 val (20% learn / 80%
# cal) + 25.000 test. Yang DIKURANGI hanya seed: 3, bukan 10.
NUM_VAL, NUM_TEST = 25000, 25000
NUM_SEEDS  = 3
ALPHAS     = '0.01 0.05 0.1'      # gratis: Phase B, numpy
SCORES     = 'thr aps raps'       # gratis
METHODS    = 'baseline tta_avg tta_learned umtta'
BATCH_SIZE = 128
SAVE_DIR   = '/content/results/imagenet_resnet50'
# =======================================================================
print('seed', NUM_SEEDS, '| val', NUM_VAL, '| test', NUM_TEST)

## 2. Drive, repo, dan verifikasi kode

Repo di-clone dari `origin/main` — commit terbaru (`cvpr revision` →
`fix: corruption pipeline`). Berkas yang dibutuhkan **diperiksa ada**, bukan
diasumsikan; kalau `unified_eval.py` hilang, lebih baik gagal di sini daripada
setelah 6,7 GB terunduh.

In [ ]:
import os, subprocess, sys, glob, time
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.isdir(UMTTA_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', UMTTA_URL, UMTTA_DIR],
                   check=True)
print(subprocess.run(['git', 'log', '--oneline', '-1'], cwd=UMTTA_DIR,
                     capture_output=True, text=True).stdout.strip())

NEED = ['unified_eval.py', 'unified_eval_corruption.py', 'score_functions.py',
        'conformal_engine.py', 'conditional_coverage.py', 'statistical_tests.py']
missing = [f for f in NEED if not os.path.exists(os.path.join(UMTTA_DIR, f))]
assert not missing, 'berkas hilang di repo: ' + repr(missing)
print('berkas inti lengkap:', len(NEED))
subprocess.run(['pip', 'install', '-q', '-r',
                os.path.join(UMTTA_DIR, 'requirements.txt')], check=False)
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

## 3. Unduh ImageNet val + label sinset

`ILSVRC2012_img_val.tar` isinya **50.000 JPEG datar tanpa folder kelas**, sedangkan
`unified_eval.py` memakai `torchvision.datasets.ImageFolder` yang **butuh subfolder
per kelas**. Berkas label sinset menutup celah itu: 50.000 WNID berurutan sesuai
nomor berkas, jadi penataannya deterministik dan bisa diverifikasi.

Keduanya dicek ukurannya setelah diunduh, bukan dipercaya begitu saja — pola yang
sudah menyelamatkan notebook 05 dari unduhan gagal yang mengembalikan exit 0.

In [ ]:
os.makedirs('/content/dl', exist_ok=True)
TAR = '/content/dl/ILSVRC2012_img_val.tar'
LBL = '/content/dl/val_synset_labels.txt'

if not os.path.exists(LBL) or os.path.getsize(LBL) < 100000:
    subprocess.run(['wget', '-q', LBL_URL, '-O', LBL], check=True)
lbls = [l.strip() for l in open(LBL) if l.strip()]
assert len(lbls) == 50000, 'label harus 50.000 baris, dapat ' + str(len(lbls))
print('label OK:', len(lbls), 'baris |', len(set(lbls)), 'WNID unik')

done = len(glob.glob(VAL_DIR + '/*/*.JPEG'))
if done >= 50000:
    print('val sudah tertata:', done, 'gambar -- unduhan & ekstraksi dilewati')
elif not os.path.exists(TAR) or os.path.getsize(TAR) < 6_000_000_000:
    subprocess.run(['apt-get', 'install', '-qq', 'aria2'], check=False)
    t0 = time.time()
    subprocess.run(['aria2c', '-x', '16', '-s', '16', VAL_TAR,
                    '-d', '/content/dl', '-o', 'ILSVRC2012_img_val.tar'],
                   check=True)
    print('unduh selesai {:.0f}s | {:.2f} GB'.format(
        time.time() - t0, os.path.getsize(TAR) / 1e9))
else:
    print('tar sudah ada: {:.2f} GB'.format(os.path.getsize(TAR) / 1e9))

## 4. Tata ke folder WNID — lalu VERIFIKASI

1.000 subfolder, **tepat 50 gambar tiap folder**, total 50.000. Kalau salah satu
meleset, seluruh eksperimen di atasnya tidak sah, jadi ini di-assert bukan dicetak.
Tar dihapus setelah berhasil supaya disk tidak menahan 6,7 GB percuma.

In [ ]:
if len(glob.glob(VAL_DIR + '/*/*.JPEG')) < 50000:
    RAW = '/content/dl/raw_val'
    os.makedirs(RAW, exist_ok=True)
    if len(glob.glob(RAW + '/*.JPEG')) < 50000:
        t0 = time.time()
        subprocess.run(['tar', '-xf', TAR, '-C', RAW], check=True)
        print('ekstrak {:.0f}s'.format(time.time() - t0))
    os.makedirs(VAL_DIR, exist_ok=True)
    for w in set(lbls):
        os.makedirs(os.path.join(VAL_DIR, w), exist_ok=True)
    moved = 0
    for i, w in enumerate(lbls, start=1):
        src = os.path.join(RAW, 'ILSVRC2012_val_%08d.JPEG' % i)
        if os.path.exists(src):
            os.rename(src, os.path.join(VAL_DIR, w,
                                        'ILSVRC2012_val_%08d.JPEG' % i))
            moved += 1
    print('dipindah', moved)

dirs = sorted(d for d in os.listdir(VAL_DIR)
              if os.path.isdir(os.path.join(VAL_DIR, d)))
counts = [len(os.listdir(os.path.join(VAL_DIR, d))) for d in dirs]
total = sum(counts)
print('folder', len(dirs), '| total', total,
      '| per folder min', min(counts), 'max', max(counts))
assert len(dirs) == 1000, 'harus 1000 kelas, dapat ' + str(len(dirs))
assert total == 50000, 'harus 50.000 gambar, dapat ' + str(total)
assert min(counts) == max(counts) == 50, 'tiap kelas harus 50 gambar'
print('VERIFIKASI LULUS -- struktur ImageFolder sah')
for p in (TAR, '/content/dl/raw_val'):
    if os.path.exists(p):
        subprocess.run(['rm', '-rf', p], check=False)
print('tar & raw dihapus, disk dibebaskan')

## 5. Phase A — ekstrak + cache logit (GPU)

Ini satu-satunya bagian yang butuh GPU, dan satu-satunya yang mahal. Hasilnya cache
logit per metode per seed; setelah ini α, fungsi skor, dan **PCC** semuanya jalan
di atasnya tanpa GPU sama sekali.

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
cmd = [sys.executable, 'unified_eval.py',
       '--data', VAL_DIR, '--dataset', 'imagenet',
       '--num_val', str(NUM_VAL), '--num_test', str(NUM_TEST),
       '--num_seeds', str(NUM_SEEDS),
       '--alpha'] + ALPHAS.split() + ['--scores'] + SCORES.split() + \
      ['--methods'] + METHODS.split() + \
      ['--batch_size', str(BATCH_SIZE), '--save_dir', SAVE_DIR, '--phase_a_only']
print(' '.join(cmd))
t0 = time.time()
p = subprocess.run(cmd, cwd=UMTTA_DIR)
print('Phase A rc={} dalam {:.0f}s'.format(p.returncode, time.time() - t0))
cached = glob.glob(SAVE_DIR + '/**/*.pt', recursive=True)
print('berkas cache:', len(cached))
assert p.returncode == 0 and cached, 'Phase A gagal -- jangan lanjut'

## 6. Phase B — semua α × skor × metode, tanpa GPU

In [ ]:
cmd = [sys.executable, 'unified_eval.py', '--data', VAL_DIR,
       '--dataset', 'imagenet', '--save_dir', SAVE_DIR, '--eval_only']
t0 = time.time()
p = subprocess.run(cmd, cwd=UMTTA_DIR)
print('Phase B rc={} dalam {:.0f}s'.format(p.returncode, time.time() - t0))
for f in sorted(glob.glob(SAVE_DIR + '/*.json'))[:10]:
    print('  ', os.path.basename(f), '{:.0f} KB'.format(os.path.getsize(f)/1e3))

## 6b. SIMPAN SEGERA setelah Phase A

Penyalinan dipanggil di sini **dan** di akhir. Alasannya konkret: Phase A adalah
satu-satunya bagian yang butuh GPU dan bisa memakan puluhan menit; kalau runtime
mati di Phase B atau saat menyalin, kerja GPU itu hilang seluruhnya. Menyalin dua
kali murah; kehilangan Phase A tidak.

Repo juga memeriksa cache sebelum menghitung, jadi run berikutnya yang menemukan
cache di Drive bisa melewati Phase A sepenuhnya.

In [ ]:
import shutil
DEST = DRIVE_ROOT + '/umtta/imagenet_resnet50'

def save_to_drive(tag):
    os.makedirs(DEST, exist_ok=True)
    n_ok = n_bad = 0
    for p in glob.glob(SAVE_DIR + '/**/*', recursive=True):
        if not os.path.isfile(p):
            continue
        d = os.path.join(DEST, os.path.relpath(p, SAVE_DIR))
        os.makedirs(os.path.dirname(d), exist_ok=True)
        try:
            if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(p):
                n_ok += 1          # sudah tersalin utuh, lewati
                continue
            shutil.copy2(p, d)
            n_ok += 1 if os.path.getsize(d) == os.path.getsize(p) else 0
            n_bad += 0 if os.path.getsize(d) == os.path.getsize(p) else 1
        except Exception as e:
            n_bad += 1
            print('  gagal', os.path.relpath(p, SAVE_DIR), e)
    tot = sum(os.path.getsize(f) for f in glob.glob(DEST + '/**/*', recursive=True)
              if os.path.isfile(f))
    print('[{}] tersalin {} gagal {} | {:.2f} GB di Drive'.format(
        tag, n_ok, n_bad, tot / 1e9))
    print('    ' + ('AMAN' if n_bad == 0 else 'ADA YANG GAGAL -- jangan hapus runtime'))
    return n_bad == 0

save_to_drive('setelah Phase A')

## 7. Simpan ke Drive — cache logit ikut

Cache logit-nya yang paling berharga: ia yang membuat PCC bisa dijalankan di atas
backbone ini **tanpa GPU**, dan membuat run berikutnya tidak perlu mengulang Phase A.
Ukurannya diverifikasi setelah disalin.

In [ ]:
save_to_drive('final')
print()
print('Cache logit ada di Drive. PCC bisa jalan di atas backbone ini TANPA GPU,')
print('dan run berikutnya melewati Phase A sepenuhnya.')